In [1]:
1+1

2

## document loader

In [40]:
from langchain_community.document_loaders import PyPDFDirectoryLoader,PyPDFLoader
from src.loging.logger import log
from src.exceptions.custom_exceptions import CustomException

logging=log



class DocumentLoader:
    def __init__(self,directory:str,):
        self.directory=directory
        logging.info("initialized docuemnt loader")

    def document_loader(self):
        rew_documents=PyPDFDirectoryLoader(self.directory)
        self.docs=rew_documents.load()
        if self.docs:
            try:
                pdf_files = set()
                for doc in self.docs:
                    source = doc.metadata.get('source', 'unknown')
                    pdf_files.add(source)
                # Log each PDF file
                logging.info(f"✅ Successfully loaded {len(self.docs)}  , documents lent is  {len(pdf_files)} PDF files:")
                for pdf in pdf_files:
                    logging.info(f" 📄 {pdf}")
                
            except Exception as e:
                    logging.error(f"error in loading documents : {e}")
                    raise CustomException(
                        message=f"Failed to load documents {e}",
                        error_detail=e
                    )
        elif not self.docs:
            logging.error(f" No documents found in {self.directory}")
            print(f"⚠️ No documents found in {self.directory}")
         
        return self.docs
    





In [41]:
data = DocumentLoader("../data/")

2026-05-21 15:40:44,186 - INFO - initialized docuemnt loader


In [42]:
docs=data.document_loader()

2026-05-21 15:40:44,551 - WARNING - incorrect startxref pointer(1)
2026-05-21 15:40:44,554 - WARNING - parsing for Object Streams
2026-05-21 15:40:44,647 - WARNING - Error -3 while decompressing data: incorrect header check
2026-05-21 15:40:44,655 - WARNING - Error -3 while decompressing data: incorrect header check
2026-05-21 15:40:44,669 - WARNING - Error -3 while decompressing data: incorrect header check
2026-05-21 15:40:44,689 - WARNING - Error -3 while decompressing data: incorrect header check
2026-05-21 15:40:44,706 - WARNING - Error -3 while decompressing data: incorrect header check
2026-05-21 15:40:44,722 - WARNING - Error -3 while decompressing data: incorrect header check
2026-05-21 15:40:44,742 - WARNING - Error -3 while decompressing data: incorrect header check
2026-05-21 15:40:44,760 - WARNING - Error -3 while decompressing data: incorrect header check
2026-05-21 15:40:44,780 - WARNING - Error -3 while decompressing data: incorrect header check
2026-05-21 15:40:44,802 

In [44]:
docs[0].metadata

{'producer': 'www.ilovepdf.com',
 'creator': 'Microsoft® Word 2016',
 'creationdate': '2021-02-16T14:31:27+00:00',
 'title': 'Plan cptable Maroc.PDF',
 'author': 'Pierre',
 'moddate': '2021-02-16T14:31:31+00:00',
 'source': '..\\data\\Plan_Comptable_marocain.pdf',
 'total_pages': 24,
 'page': 0,
 'page_label': '1'}

In [43]:
docs[0].metadata.get("source")

'..\\data\\Plan_Comptable_marocain.pdf'

## embedings

In [54]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from src.loging.logger import log
from src.exceptions.custom_exceptions import CustomException
import sys

logging=log


class Embeddings:

    def __init__(self,
        model_name:str="sentence-transformers/all-MiniLM-L6-v2",
        device: str = "cpu",
        normalize_embeddings: bool = True,
        batch_size: int = 32):

        self.model_name = model_name
        self.device = device
        self.normalize_embeddings = normalize_embeddings
        self.batch_size = batch_size
        self._embeddings = None
        logging.info("embedding get initialized")


    def initializing_embedding(self):
        if self._embeddings is None:
            try:
                self.embeddings = HuggingFaceEmbeddings(
                        model_name=self.model_name,
                        model_kwargs={'device': self.device},
                        encode_kwargs={
                            'normalize_embeddings': self.normalize_embeddings,
                            'batch_size': self.batch_size
                        }
                    )
                log.info(f"initialiased embeding model {HuggingFaceEmbeddings.__class__.__name__} with model name : {self.model_name}")
            except Exception as e:
                    logging.error(f"error during initilizing embedings : {e}")
                    raise CustomException(
                        f"Failed to initialize embeddings with model {self.model_name} or their is an error in embeding models {e}",
                        sys
                    )
        return self.embeddings

## chunker

In [66]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker   # ← FIXED
from langchain_community.document_loaders import PyPDFDirectoryLoader
from src.loging.logger import log
from src.exceptions.custom_exceptions import CustomException

logging=log



class TextSpliter:
    def __init__(self,embeding_model,persist_directory:str="vectorestore_VDB"):
        self.embeding=embeding_model
        self.persist_directory=persist_directory
        


    
    def split_documents(self,documents):
        # Split
        try:
            text_splitter = SemanticChunker(
                            embeddings=self.embeding,
                            breakpoint_threshold_type="percentile"
                        )

           
            chunks = text_splitter.split_documents(documents)
            logging.info(f"✅ Split {len(documents)} documents into {len(chunks)} chunks")
            return chunks
        except Exception as e:
            logging.error(f"❌ Error splitting documents: {e}")
            raise CustomException(f"Failed to chunk  {e}",sys)






## retriver

In [73]:
from langchain_chroma import Chroma
from src.loging.logger import log
from src.exceptions.custom_exceptions import CustomException

logging=log
from pathlib import Path

class VectorStore:
    def __init__(self, embeddings, persist_directory):
        self.embeddings = embeddings
        self.persist_directory = persist_directory
        self.vectorstore = None
        Path(persist_directory).mkdir(parents=True, exist_ok=True)
        logging.info(f"✅ VectorStore initialized with persist_dir: {persist_directory}")

    def create_from_documents(self, documents):
        """Create vectorstore from documents"""
        try:
            self.vectorstore = Chroma.from_documents(
                documents=documents,
                embedding=self.embeddings,
                persist_directory=self.persist_directory
            )
            logging.info(f"✅ Created vectorstore with {len(documents)} documents")
            return self.vectorstore
        except Exception as e:
            logging.error(f"❌ Error creating vectorstore: {e}")
            raise CustomException("Failed to create vectorstore", e)

    def load_existing(self):
        """Load existing vectorstore"""
        try:
            self.vectorstore = Chroma(
                embedding_function=self.embeddings,
                persist_directory=self.persist_directory
            )
            logging.info(f"✅ Loaded existing vectorstore from {self.persist_directory}")
            print(f"✅ Loaded existing vectorstore from {self.persist_directory}")
            return self.vectorstore
        except Exception as e:
            logging.error(f"❌ Error loading vectorstore: {e}")
            raise CustomException("Failed to load vectorstore", e)

    def get_retriever(self, k: int = 4):
        """Get retriever from vectorstore"""
        if self.vectorstore is None:
            raise CustomException("Vectorstore not created yet. Call create_from_documents first.")
        return self.vectorstore.as_retriever(search_type="mmr",
                                             search_kwargs={"k": k,
                                                            "fetch_k": 20,
                                                            "lambda_mult": 0.5 })

In [110]:
data = DocumentLoader("../data/PLAN_COMPTABLE")
docs=data.document_loader()
embeding_model=Embeddings()
embeding=embeding_model.initializing_embedding()
text_splietr=TextSpliter(embeding)
chunks=text_splietr.split_documents(docs)
vectore_store=VectorStore(embeding,"artifacts/vectorestore/CGI")
vectore_store.create_from_documents(chunks)
store=vectore_store.get_retriever()

2026-05-21 20:01:24,743 - INFO - initialized docuemnt loader
2026-05-21 20:01:29,357 - INFO - ✅ Successfully loaded 24  , documents lent is  1 PDF files:
2026-05-21 20:01:29,357 - INFO -  📄 ..\data\PLAN_COMPTABLE\Plan_Comptable_marocain.pdf
2026-05-21 20:01:29,374 - INFO - embedding get initialized
2026-05-21 20:01:29,622 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:01:29,668 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-05-21 20:01:29,831 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:01:29,877 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformer

In [112]:
print(chunks[2].page_content)

16 COMPTES DE LIAISON DES ETABLISSEMENTS ET SUCCURSALES 
160 Comptes de liaison des établissements et succursales 
1601 Comptes de liaison du siège 
1605 Comptes de liaison des établissements 
17 Ecarts de conversion - Passif 
171 Augmentation des créances immobilisées 
1710 Augmentation des créances immobilisées 
172 Diminution des dettes de financement 
1720 Diminution des dettes de financement 
 
 
 
 
CLASSE 2 : COMPTES D'ACTIF IMMOBILISE 
 
21 IMMOBILISATIONS EN NON-VALEURS 
211 Frais préliminaires 
2111 Frais de constitution 
2112 Frais préalables au démarrage 
2113 Frais d'augmentation du capital 
2114 Frais sur opérations de fusions, scissions et transformations 
2116 Frais de prospection 
2117 Frais de publicité 
2118 Autres frais préliminaires 
212 Charges à répartir sur plusieurs exercices 
2121 Frais d'acquisitition des immobilisations 
2125 Frais d'émission des emprunts 
2128 Autres charges à répartir 
213 Primes de remboursement des obligations 
2130 Primes de rembourseme

In [113]:
store.invoke("tva recuperable sur charge")

[Document(id='e4314ba6-332c-42c2-bb95-c95943fe746c', metadata={'page_label': '172', 'moddate': '2025-12-30T13:12:54+01:00', 'page': 171, 'author': 'SADELLAH Ouiam', 'creator': 'Microsoft® Word 2016', 'source': '..\\data\\Code_impôts_2026.pdf', 'producer': 'Microsoft® Word 2016', 'total_pages': 765, 'creationdate': '2025-12-30T13:12:54+01:00'}, page_content='Au vu de cette lettre, le ministre chargé des finances ou la personne \ndéléguée par lui à cet effet, établit un ordre de recette au nom du notaire \naccompagné du chèque cité ci -dessus permettant au receve ur de \nl’administration fiscale la récupération du montant de la taxe sur la valeur \najoutée ; \n \n6°-La mainlevée de l’hypothèque ne peut être délivrée qu’après \nproduction par l’intéressé : \n \n- du contrat définitif du transfert de propriété ;'),
 Document(id='ea02e9d4-e641-4e75-bb4a-a753e15a8f9a', metadata={'producer': 'Microsoft® Word 2016', 'author': 'SADELLAH Ouiam', 'creationdate': '2025-12-30T13:12:54+01:00', 'mod

In [114]:
from src.PipeLine.pipeline import  RagPipeLine

In [115]:
pipeline=RagPipeLine(data_dir="../data/PLAN_COMPTABLE",persist_dir="artifacts/vectorestore/plan_comptable",force_rebuild=True)


🚀 RAG Pipeline initialized
📂 Data: ../data/PLAN_COMPTABLE
💾 Persist: artifacts/vectorestore/plan_comptable
🔍 Vectorstore exists: False



In [116]:
pip=pipeline.run()

2026-05-21 20:01:57,213 - INFO - embedding get initialized
2026-05-21 20:01:57,502 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:01:57,569 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-05-21 20:01:57,736 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:01:57,784 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-21 20:01:57,791 - INFO - Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-05-21 20:01:

📄 Loaded 24 documents from cgnc folder
🔤 Embeddings ready


--- Logging error ---
Traceback (most recent call last):
  File "C:\dev\aaa\src\PipeLine\pipeline.py", line 55, in run
    print("❌ No documents to process. Please add PDF files to data/CGNC/")
        ^^^^^^^^^^^^^^^^
AttributeError: 'RagPipeLine' object has no attribute 'text_splietrs'. Did you mean: 'text_spliter'?

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\dev\aaa\env\Lib\logging\__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "c:\dev\aaa\env\Lib\logging\__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "c:\dev\aaa\env\Lib\logging\__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "c:\dev\aaa\env\Lib\logging\__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted dur

In [117]:
pip.invoke("les cadaux publicitaire dans  IR")

AttributeError: 'NoneType' object has no attribute 'invoke'

In [ ]:
pip